# 1 — Data Cleaning
Inspect and clean each dataset independently.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
D=ROOT/"data"
files={"churn":"customer_churn.csv","sales":"sales_data.csv","housing":"house_prices.csv"}
for name,file in files.items():
 df=pd.read_csv(D/file); print(name,df.shape); display(df.isna().sum(),df.head())

In [ ]:
c=pd.read_csv(D/"customer_churn.csv").drop_duplicates("CustomerID")
for col in ["Tenure","MonthlyCharges","TotalCharges","SeniorCitizen","Churn"]: c[col]=pd.to_numeric(c[col],errors="coerce")
c.to_csv(D/"cleaned_data.csv",index=False)
h=pd.read_csv(D/"house_prices.csv").drop_duplicates("Property_ID"); h.to_csv(D/"cleaned_house_prices.csv",index=False)
s=pd.read_csv(D/"sales_data.csv").drop_duplicates(); s["Date"]=pd.to_datetime(s.Date,errors="coerce"); s.to_csv(D/"cleaned_sales_data.csv",index=False)
print("Three cleaned files saved")

# 2 — Exploratory Data Analysis

In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
D=ROOT/"data"
import matplotlib.pyplot as plt
c=pd.read_csv(D/"cleaned_data.csv"); s=pd.read_csv(D/"cleaned_sales_data.csv",parse_dates=["Date"]); h=pd.read_csv(D/"cleaned_house_prices.csv")
print("Churn rate",f"{c.Churn.mean():.1%}"); display(c.groupby("Contract").Churn.mean())
display(s.groupby("Region").Total_Sales.sum()); display(h.groupby("Location").Price.median())

In [ ]:
c.groupby("Contract").Churn.mean().plot.bar(title="Churn by contract"); plt.show()
s.groupby("Region").Total_Sales.sum().plot.bar(title="Sales by region"); plt.show()
s.groupby("Date").Total_Sales.sum().plot(title="Daily sales"); plt.show()
plt.scatter(h.Area,h.Price); plt.title("Area vs price"); plt.show()
h.groupby("Location").Price.median().plot.bar(title="Median price by location"); plt.show()

# 3 — Analysis and Recommendations

In [ ]:
from pathlib import Path
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
D=ROOT/"data"
from scipy.stats import chi2_contingency,mannwhitneyu
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
c=pd.read_csv(D/"cleaned_data.csv"); h=pd.read_csv(D/"cleaned_house_prices.csv")
for col in ["Contract","PaymentMethod"]:
 stat,p,df,_=chi2_contingency(pd.crosstab(c[col],c.Churn)); print(col,"chi-square p=",p)
for col in ["Tenure","MonthlyCharges"]:
 print(col,"Mann-Whitney p=",mannwhitneyu(c.loc[c.Churn==0,col],c.loc[c.Churn==1,col]).pvalue)

In [ ]:
X=pd.get_dummies(h.drop(columns=["Property_ID","Price"]),drop_first=True,dtype=float); y=h.Price
a,b,ya,yb=train_test_split(X,y,test_size=.2,random_state=42); model=LinearRegression().fit(a,ya); pred=model.predict(b)
print("MAE",mean_absolute_error(yb,pred),"MSE",mean_squared_error(yb,pred),"R2",r2_score(yb,pred))